# 시작 전 

- 실습은 노트북 파일을 사용합니다. 사용하는 IDE에 맞춰 확장 프로그램을 설치하시거나 주피터랩 혹은 주피터 노트북을 설치해주시면 감사하겠습니다.
- 노트북을 실행하기 위한 커널 또한 준비해주셔야 합니다.
```
# 가상환경을 사용할 경우
# $ conda activate carla가상환경이름
$ pip install ipykernel
$ python -m ipykernel install --user --name carla_env --display-name "carla_0.9.14"
```

# CARLA 서버 설치

본 워크숍에서는 0.9.14 버전을 사용합니다.
운영 체제에 맞춰 파일을 다운 받아 주세요.

[CARLA_0.9.14 다운로드](https://github.com/carla-simulator/carla/releases/tag/0.9.14/)

CARLA 0.9.14 서버를 직접 실행하기 위해서는 다음의 사양을 체크해주세요.

- 운영 체제 : 윈도우 10, 11 / 우분투 20.04, 22.04 
- RTX2070 이상의 GPU 혹은 8GB 이상의 VRAM을 가진 GPU
- 20GB 이상의 디스크 공간
- CARLA는 기본적으로 2000번과 2001 포트를 사용하므로, 해당 포트가 방화벽이나 다른 프로그램에 의해 차단되지 않도록 해주세요 .
- Python 3.7
- PIP 20.3 이상

# CARLA 서버 실행
```
# carla 설치 폴더로 이동
$ ./CarlaUE4.sh
# 윈도우의 경우 CarlaUE4.exe 실행
```

# CALRLA 클라이언트 라이브러리 설치
wheel 파일 직접 설치

```
cd /PythonAPI/carla/dist/
pip install carla-0.9.14-cp37-cp37m-manylinux_2_27_x86_64.whl

# libomp5 오류 발생 시
# sudo apt-get install libomp5
```

# Carla 서버 제어
### 1. carla.Client
Carla : 서버-클라이언트 구조를 기반으로 한 오픈 소스 자율 주행 시뮬레이터.

서버의 world를 조정하거나 객체를 생성하고 이들을 다루기 위해선 carla.Client 객체를 생성해야 합니다.
주요 생성자는 다음과 같습니다. init(host=127.0.0.1, port=2000). 기본적으로 로컬 호스트와 2000번 포트에 연결하도록 설정이 되어있습니다.

    - host : Carla 시뮬레이터 서버가 작동하고 있는 host의 IP.
    - port : Calra 시뮬레이터 서버가 작동하고 있는 host의 TCP 포트.



In [1]:
import carla
client = carla.Client('localhost', 2000)


### 2. carla.Client()를 이용한 시뮬레이션 환경 설정

Carla는 다양한 시나리오를 고려할 수 있도록 잘 구축된 몇 개의 맵들을 제공합니다.
설치 파일의 크기를 위해 일부 맵은 추가 Asset으로 제공하며, 서버 파일과 마찬가지로 Carla 깃허브 저장소에서 다운받을 수 있습니다.

Carla 시뮬레이터 서버의 맵을 변경하기 위해선 carla.Client.load_world('맵이름') 함수를 사용합니다.
사용 가능한 맵 목록을 알고 싶다면 carla.Client.get_available_maps() 함수를 사용합니다.

비오는 상황 같이 기후적 시나리오도 고려할 수 있도록 날씨의 변경도 가능한데, 이 또한 Client 객체를 사용하여 가능합니다.


- carla.Client().get_world() : 현재 Carla 시뮬레이터 서버의 맵, 날씨, Actor 등에 접근할 수 있는 객체를 반환.
- carla.Client().get_available_maps() : 현재 시뮬레이터 서버에서 사용 가능한 맵 리스트 반환.
- carla.Client().load_world('맵 이름') : 현재 시뮬레이터 서버의 맵을 변경.

In [2]:
client.get_available_maps()


['/Game/Carla/Maps/Town05_Opt',
 '/Game/Carla/Maps/Town10HD',
 '/Game/Carla/Maps/Town04',
 '/Game/Carla/Maps/Town03_Opt',
 '/Game/Carla/Maps/Town04_Opt',
 '/Game/Carla/Maps/Town02_Opt',
 '/Game/Carla/Maps/Town03',
 '/Game/Carla/Maps/Town02',
 '/Game/Carla/Maps/Town01_Opt',
 '/Game/Carla/Maps/Town05',
 '/Game/Carla/Maps/Town10HD_Opt',
 '/Game/Carla/Maps/Town01']

In [3]:
client.load_world('Town04')


In [4]:
world = client.get_world()
world.set_weather(carla.WeatherParameters.MidRainSunset)

# CARLA의 Actor와 Blueprint
시뮬레이션 내에서 역할을 수행하는 모든 것들은 carla.Actor로 정의됩니다. 

차량뿐만 아니라 보행자, 신호등, 차량의 센서, 표지판 마저 모두 Actor에 속합니다.

그러나 이들 객체를 직접 생성하는 것은 아니며, blueprint를 활용하여 Actor를 생성합니다.

blueprint는 Carla에서 정의해둔, 말 그대로 블루 프린트 입니다. 각 blueprint는 시뮬레이션에서 표현될 그래픽 요소와, 해당 객체의 적절한 기능.. 예를 들어 차량 색상, role_name, 라이다 센서의 채널 수, 보행자의 속도 등의 속성들을 사전 정의해둡니다.

이러한 blueprint는 blueprint library에서 조회할 수 있습니다.

In [5]:
for vehicle in world.get_blueprint_library().filter('vehicle.*'):
    print(vehicle.id)

vehicle.audi.a2
vehicle.citroen.c3
vehicle.chevrolet.impala
vehicle.dodge.charger_police_2020
vehicle.micro.microlino
vehicle.dodge.charger_police
vehicle.audi.tt
vehicle.jeep.wrangler_rubicon
vehicle.mercedes.coupe
vehicle.mercedes.coupe_2020
vehicle.harley-davidson.low_rider
vehicle.dodge.charger_2020
vehicle.ford.ambulance
vehicle.lincoln.mkz_2020
vehicle.mini.cooper_s_2021
vehicle.toyota.prius
vehicle.ford.crown
vehicle.carlamotors.carlacola
vehicle.vespa.zx125
vehicle.nissan.patrol_2021
vehicle.mercedes.sprinter
vehicle.audi.etron
vehicle.seat.leon
vehicle.volkswagen.t2_2021
vehicle.tesla.cybertruck
vehicle.lincoln.mkz_2017
vehicle.ford.mustang
vehicle.carlamotors.firetruck
vehicle.volkswagen.t2
vehicle.mitsubishi.fusorosa
vehicle.tesla.model3
vehicle.diamondback.century
vehicle.gazelle.omafiets
vehicle.bmw.grandtourer
vehicle.bh.crossbike
vehicle.kawasaki.ninja
vehicle.nissan.patrol
vehicle.nissan.micra
vehicle.mini.cooper_s
vehicle.yamaha.yzf


In [6]:
# 수정 가능한 속성의 경우 is_modifiable를 사용하여 다음과 같이 확인할 수 있습니다. 
for attr in world.get_blueprint_library().find('vehicle.dodge.charger_police'):
    if attr.is_modifiable:
        # 또한 CARLA에서 권장하는 값도 확인할 수 있습니다.
        print(f'속성 : {attr} 권장값 : {attr.recommended_values}\n')
        
        
    

속성 : ActorAttribute(id=terramechanics,type=bool,value=False) 권장값 : ['false']

속성 : ActorAttribute(id=color,type=Color,value=Color(0,0,0,255)) 권장값 : ['0,0,0', '8,53,0']

속성 : ActorAttribute(id=sticky_control,type=bool,value=True) 권장값 : ['true']

속성 : ActorAttribute(id=role_name,type=str,value=autopilot) 권장값 : ['autopilot', 'scenario', 'ego_vehicle']



In [7]:
vehicle_bp = world.get_blueprint_library().filter('vehicle.dodge.charger_police')
print(type(vehicle_bp)) # class 'carla.libcarla.BlueprintLibrary'
pos = carla.Transform(carla.Location(x=160, y=14, z=10), carla.Rotation(pitch=0, yaw=180, roll=0)) # spawn 위치 직접 설정
actor1 = world.spawn_actor(vehicle_bp[0], pos) # param1: blueprint, param2: transform

<class 'carla.libcarla.BlueprintLibrary'>


In [8]:
# 충돌이 있는 경우 Actor는 생성되지 않습니다.
# 이러한 충돌을 방지하고 안전하게 생성하고 싶다면, 추천 스폰 지점 목록을 활용할 수 있습니다.
spawn_points = world.get_map().get_spawn_points()
print(spawn_points.pop())
# actor2 = world.spawn_actor(vehicle_bp[0], spawn_points[0])

Transform(Location(x=377.414825, y=6.316813, z=0.281942), Rotation(pitch=0.000000, yaw=134.097794, roll=0.000000))


In [8]:
# 혹은 관전자 시점에 Actor를 소환하고 싶다면..
spawn_points = world.get_spectator().get_transform()
actor4 = world.spawn_actor(vehicle_bp[0], spawn_points)

In [22]:
# 보행자를 생성하고 싶다면..
walker_bp_library = world.get_blueprint_library().filter('walker.pedestrian.*')
print(walker_bp_library)

walker_bp = walker_bp_library.find('walker.pedestrian.0013')

spawn_points_wk = world.get_spectator().get_transform()
walker = world.spawn_actor(walker_bp, spawn_points_wk)


[ActorBlueprint(id=walker.pedestrian.0013,tags=[0013, pedestrian, walker]), ActorBlueprint(id=walker.pedestrian.0027,tags=[0027, pedestrian, walker]), ActorBlueprint(id=walker.pedestrian.0008,tags=[0008, pedestrian, walker]), ActorBlueprint(id=walker.pedestrian.0006,tags=[pedestrian, 0006, walker]), ActorBlueprint(id=walker.pedestrian.0035,tags=[0035, pedestrian, walker]), ActorBlueprint(id=walker.pedestrian.0001,tags=[0001, pedestrian, walker]), ActorBlueprint(id=walker.pedestrian.0002,tags=[0002, pedestrian, walker]), ActorBlueprint(id=walker.pedestrian.0003,tags=[0003, pedestrian, walker]), ActorBlueprint(id=walker.pedestrian.0004,tags=[0004, pedestrian, walker]), ActorBlueprint(id=walker.pedestrian.0011,tags=[0011, pedestrian, walker]), ActorBlueprint(id=walker.pedestrian.0037,tags=[0037, pedestrian, walker]), ActorBlueprint(id=walker.pedestrian.0005,tags=[0005, pedestrian, walker]), ActorBlueprint(id=walker.pedestrian.0009,tags=[0009, pedestrian, walker]), ActorBlueprint(id=walker

# 센서 데이터 및 수집

carla.Sensor 클래스는 데이터를 측정하고 스트리밍할 수 있는 액터 클래스 입니다.
데이터 유형은 carla.SensorData이며 listen() 함수를 통해 데이터를 수신 및 관리합니다.



In [ ]:
# 카메라 센서 설정
camera_bp = world.get_blueprint_library().filter('sensor.camera.rgb')[0] # blueprint 라이브러리에서 카메라 센서의 blueprint를 가져오기
# 카메라 센서의 경우 image_size_x, image_size_y, 시야각 등의 속성을 설정할 수 있습니다.
camera_bp.set_attribute('image_size_x', '1920')
camera_bp.set_attribute('image_size_y', '1080')
camera_bp.set_attribute('fov', '110')

camera_bp.set_attribute('sensor_tick', '5.0') # 센서가 데이터를 생성하는 간격을 설정하는 속성입니다. 5.0으로 설정하면 센서가 5초마다 데이터를 생성합니다.

# 카메라 센서 부착
camera_transform = carla.Transform(carla.Location(x=0, y=0, z=2), carla.Rotation(pitch=0, yaw=0, roll=0)) # 카메라의 위치와 회전을 정의하는 Transform 객체 생성
camera = world.spawn_actor(camera_bp, camera_transform, attach_to=actor4, attachment_type=carla.AttachmentType.Rigid) # 카메라 센서를 actor4에 부착. 카메라의 transform은 부착할 actor에 기반한 위치. attach_type은 Rigid로 설정하여 카메라가 차량과 완전히 고정되도록 합니다.

In [11]:

# 센서 데이터 수집 : 모든 센서에는 listen()메서드가 있습니다. 이 메서드는 센서가 데이터를 가져올 때마다 호출됩니다.
# camera.listen(lambda image: image.save_to_disk('output/%06d.png' % image.frame)) # 카메라 센서가 캡처한 이미지를 저장하는 콜백 함수 등록

camera.stop() # 카메라 센서의 데이터 수집 중지

In [12]:
# 시멘틱 세그멘테이션 카메라 센서 설정
semantic_bp = world.get_blueprint_library().filter('sensor.camera.semantic_segmentation')[0]
semantic_bp.set_attribute('image_size_x', '1920')
semantic_bp.set_attribute('image_size_y', '1080')
semantic_bp.set_attribute('fov', '110')
semantic_bp.set_attribute('sensor_tick', '5.0')

semantic_transform = carla.Transform(carla.Location(x=0, y=0, z=2), carla.Rotation(pitch=0, yaw=0, roll=0))
semantic_camera = world.spawn_actor(semantic_bp, camera_transform, attach_to=actor4, attachment_type=carla.AttachmentType.Rigid)




In [15]:
# 시멘틱 세그멘테이션 카메라의 데이터 수집
# 시멘틱 세그멘테이션 카메라의 경우  carla.ColorConverter의 CityScapesPalette를 활용하여 시멘틱 세그멘테이션 이미지에 색상 맵을 적용하여 저장합니다.
# semantic_camera.listen(lambda image: image.save_to_disk('output/semantic/%06d.png' % image.timestamp, carla.ColorConverter.CityScapesPalette))

# 시멘틱 세그멘테이션 카메라의 데이터 수집 중지
semantic_camera.stop() 




In [16]:
# 이외에도 라이더, 레이더, GNSS, IMU 등 다양한 센서가 존재합니다. 각 센서의 blueprint를 가져와서 원하는 속성을 설정하고, 차량에 부착하여 데이터를 수집할 수 있습니다.
sensor_bp  = world.get_blueprint_library().filter('sensor.*')
for sensor in sensor_bp:
    print(sensor.id)



sensor.other.lane_invasion
sensor.other.collision
sensor.camera.depth
sensor.lidar.ray_cast
sensor.camera.semantic_segmentation
sensor.other.radar
sensor.camera.instance_segmentation
sensor.camera.rgb
sensor.other.rss
sensor.camera.optical_flow
sensor.lidar.ray_cast_semantic
sensor.other.obstacle
sensor.other.imu
sensor.other.gnss
sensor.camera.dvs
sensor.camera.normals


# 차량의 생성 및 제어

CARLA에서 role_name=hero는 중요합니다. CARLA에서 ego_vehicle을 지정하는 속성이기 때문입니다.


뿐만 아니라 hero 차량을 중심으로 특정 반경의 물리 연산이 집중되며 CARLA에서 제공하는 녹화 기능 혹은 이벤트 로깅 기능 또한 hero 차량을 중심으로 수행되기 때문입니다.


따라서 주요 차량에 대하여 블루 프린트 생성 단계에 명시적으로 role_name='hero' 설정을 주입해야 합니다.


In [17]:
car1_bp = world.get_blueprint_library().find('vehicle.audi.tt')
car1_bp.set_attribute('role_name', 'hero') 

# 사전 제공되는 안전한 spawn 위치를 활용하여 차량을 생성합니다.
spawn_points_1 = world.get_map().get_spawn_points()



In [18]:
# 혹은 현재 서버 화면(Spectator)의 위치를 활용하여 spawn 위치를 추출합니다.
spectator = world.get_spectator()
spectator_transform = spectator.get_transform()

print(spectator_transform)

spawn_points_2 = spectator_transform
car_1 = world.spawn_actor(car1_bp, spawn_points_2)

Transform(Location(x=276.398682, y=-170.713684, z=4.246974), Rotation(pitch=-9.501250, yaw=-1.819519, roll=0.000079))


In [19]:
# 혹은 대략적인 X, Y 좌표만 알고 있을 경우 waypoint를 활용하여 차선 중앙 좌표를 추출하고 z축을 살짝 띄워 생성할 수 있습니다.
approximate_location = carla.Location(x=230, y=160)
spawn_points_3 = world.get_map().get_waypoint(approximate_location).transform
spawn_points_3.location.z += 2.0 # z축을 2.0만큼 띄워서 차량이 지면과 충돌하지 않도록 합니다.
print(spawn_points_3)

car_3 = world.spawn_actor(car1_bp, spawn_points_3)

Transform(Location(x=239.022720, y=180.823807, z=2.000000), Rotation(pitch=0.000000, yaw=-23.129292, roll=0.000000))


# 차량 제어하기 : Vehicle Controller
차량 제어의 핵심은 Vehicle Controller의 생성과 설정, 그리고 이를 차량에 적용하는 것입니다.

### 차량 조명
차량의 조명을 제어하기 위해선 carla.VehicleLightState 클래스와 Vehicle.set_light_state() 함수를 사용합니다.
차량에 따라 지원되는 조명의 종류가 다양하므로 참고 바랍니다.


In [20]:
# 좌측 깜빡이를 3초간 키는 상황 예시
import time

l_blinker = carla.VehicleLightState.LeftBlinker
off = carla.VehicleLightState.NONE
car_3.set_light_state(carla.VehicleLightState(l_blinker))
time.sleep(3)
car_3.set_light_state(carla.VehicleLightState(off))

### 차량 움직임
차량의 움직임을 제어하기 위해 carla.VehicleControl 클래스와 Vehicle.apply_control() 함수를 사용합니다.
VehicleControl 클래스엔 이동과 관련된 속성이 존재하며, throttle은 차량의 속력을, steer는 조향과 관련된 속성입니다. 이외에 brake, gear 등의 속성이 존재합니다.




In [21]:
vc = carla.VehicleControl()
vc.throttle = 0.5 # 가속 페달을 50% 밟는 상황 예시
vc.reverse = True # 후진 기어로 변경
car_3.apply_control(vc)

In [22]:
# 정지 시키고 싶다면..?
vc = carla.VehicleControl()
vc.brake =  1.0
car_3.apply_control(vc)

In [23]:
# 시나리오의 구현은 이러한 VehicleController를 적용하는 것의 반복으로 구성됩니다.
# 그러나 이러한 방법은 차량의 쓰로틀 값과 스티어 값을 직접 적용해야하므로 까다롭고 제한적입니다.

# adjacent vehicle(주변 차량)
car_bp = world.get_blueprint_library().find('vehicle.mercedes.coupe_2020')
car_bp.set_attribute('role_name','hero')
car_bp.set_attribute('color','255,255,255')

# ego vehicle(중심 차량)
ego_bp = world.get_blueprint_library().find('vehicle.mercedes.coupe_2020')

# 차량이 소환되었을 때의 Rotation: carla.Rotation(pitch=0, yaw=-179.57, roll=0)
spawn_point_A = carla.Transform(carla.Location(x=146.1, y=17.8, z=11.4), carla.Rotation(pitch=0, yaw=-179.57, roll=0))
spawn_point_B = carla.Transform(carla.Location(x=154.6, y=14.5, z=11.1), carla.Rotation(pitch=0, yaw=-179.57, roll=0))
spawn_point_C = carla.Transform(carla.Location(x=154.7, y=18, z=11.1),carla.Rotation(pitch=0, yaw=-179.57, roll=0))

car_A = world.spawn_actor(car_bp, spawn_point_A)

time.sleep(1)

vc_A = carla.VehicleControl(reverse=True)
car_A.apply_control(vc_A)

time.sleep(0.5)

vc_A = carla.VehicleControl(reverse=False)
car_A.apply_control(vc_A)

# ego vehicle
car_B = world.spawn_actor(ego_bp, spawn_point_B)

time.sleep(1)

vc_B = carla.VehicleControl(reverse=True)
car_B.apply_control(vc_B)

time.sleep(0.5)

vc_B = carla.VehicleControl(reverse=False)
car_B.apply_control(vc_B)

car_C = world.spawn_actor(car_bp, spawn_point_C)

time.sleep(1)

vc_C = carla.VehicleControl(reverse=True)
car_C.apply_control(vc_C)

time.sleep(0.5)

vc_C = carla.VehicleControl(reverse=False)
car_C.apply_control(vc_C)

v0 = carla.VehicleControl(throttle=0.7, steer=0)  # 직진
v1 = carla.VehicleControl(throttle=0.2, steer=0)
v2 = carla.VehicleControl(throttle=0.5, steer=-0.05)
v3 = carla.VehicleControl(throttle=0.5, steer=0.19)
v5 = carla.VehicleControl(throttle=0.5, steer=-0.01)
v4 = carla.VehicleControl(throttle=1, steer=0)
v6 = carla.VehicleControl(throttle=0, steer=0)  # 정지
v7 = carla.VehicleControl(throttle=0.5, steer=0.07)
v8 = carla.VehicleControl(throttle=0.5, steer=-0.30)
v9 = carla.VehicleControl(throttle=0.7, steer=-0.05)
v10 = carla.VehicleControl(throttle=0.7, steer=0.07)

l_blinker = carla.VehicleLightState.LeftBlinker  # 좌측 깜빡이
R_blinker = carla.VehicleLightState.RightBlinker  # 우측 깜빡이
nn = carla.VehicleLightState.NONE  # 점등



In [24]:
# A,B,C 차량 직진
car_A.apply_control(v0)
car_B.apply_control(v0)
car_C.apply_control(v0)
time.sleep(4)

# B차량 좌측 깜빡이
car_B.set_light_state(l_blinker)
time.sleep(1)

# C차량 감속
car_C.apply_control(v1)
time.sleep(1)

# B차량 끼어들기
car_B.apply_control(v2)
time.sleep(1)
car_B.apply_control(v3)
time.sleep(0.3)

car_B.set_light_state(nn)

# 전 차량 다시 주행
car_A.apply_control(v0)
car_B.apply_control(v0)
car_C.apply_control(v4)

# Waypoint 기반 차량 주행
Vehicle Controller를 직접 작성하여 조작하는 방식의 한계를 극복하고, 더 정교한 시나리오를 구하기 위해 Waypoint 기반 제어 방법을 사용하였습니다.

Waypoint 기반 제어의 핵심은 CARLA가 제공하는 Waypoint와 PID 컨트롤러를 활용하여 차량이 차선의 중심을 안정적으로 따라가도록 자동 제어하는 방식입니다. 이 방식은 다음의 스텝을 따릅니다.
- 경로 탐색 : 현재 차량 위치에서 가장 가까운 도로 위의 Waypoint를 찾습니다.
- 목표 설정 : 탐색한 Waypoint로부터 일정 거리 앞에 있는 다음 Waypoint를 주행 목표로 설정합니다.
- 자동 제어 : CARLA의 PID 컨트롤러가 목표 Waypoint까지 도달하기 위한 최적의 throttle, brake steer 값을 계산합니다.
- 주행 : PID 컨트롤러가 계산한 제어 값을 차량에 적용하여 목표 지점으로 주행합니다.
- 반복 : 위 과정을 지속 반복하여 경로를 이탈하지 않고 안정적인 주행을 유지합니다.